# Step 12b — Retrain, then corrected network transitions (11A) and causal interventions (12F)

The Step 7/10 checkpoints were not committed, so this notebook is self-contained. **Commit it with Save & Run All** so its `/kaggle/working/unet_*.pt` files persist for Step 13.

1. Train pairs (0,1), (3,8), (0,8): seed 0 (set `SEEDS=[0,1,2]` for error bars on the transition), EMA, wide σ range, checkpoints at epochs 20 and 80. ~8 min per model on a T4.
2. **11A.** Bisected zero-crossing transition of the network drift for every checkpoint, compared with the exact bisected transition (same method, same convention). Replaces every `tau_fold` in the paper.
3. **12F.** Causal interventions on the network's excess asymmetry, with CNN and LDA labels, deterministic and stochastic samplers.

In [ ]:
SEEDS = [0]
EPOCHS = 80
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, glob, os, time, copy
import matplotlib.pyplot as plt
from torchvision import datasets
from scipy.signal import fftconvolve
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
rng = np.random.default_rng(0); SIGMA_DATA = 0.5
d = datasets.MNIST('/kaggle/working/mnist', train=True, download=True); X_MN = (d.data.numpy().astype(np.float32)/127.5 - 1.0).reshape(-1, 784); Y_MN = d.targets.numpy()
PAIRS = [(0, 1), (3, 8), (0, 8)]

GRID = np.linspace(-4, 4, 4001); DG = GRID[1]-GRID[0]
def smooth_hist(m0, tau):
    h, _ = np.histogram(m0, bins=np.append(GRID-DG/2, GRID[-1]+DG/2), density=True); k = np.exp(-GRID**2/(2*tau)); k /= k.sum()
    return np.maximum(fftconvolve(h, k, mode='same'), 1e-300)
def exact_drift(m0, tau): p = smooth_hist(m0, tau); return GRID, np.gradient(np.log(p), GRID), p
def _merge(pts, merge=0.02):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary_points(p):
    d_ = np.gradient(np.log(p), GRID); keep = p > 1e-4*p.max()
    idx = np.where((np.sign(d_[:-1]) != np.sign(d_[1:])) & keep[:-1] & keep[1:])[0]
    return _merge([(float(GRID[i]-d_[i]*DG/(d_[i+1]-d_[i])) if d_[i+1] != d_[i] else float(GRID[i]), 'max' if d_[i] > 0 else 'min') for i in idx])
def n_max(pts): return sum(k == 'max' for _, k in pts)
def birth_bisect(get_pts, taus=np.geomspace(0.05, 3.0, 60), iters=12):
    ts = sorted(taus, reverse=True); hi = None
    for tau in ts:
        pts = get_pts(tau)
        if n_max(pts) >= 2 and hi is not None: lo = tau; break
        if n_max(pts) == 1: hi = tau
    else: return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi)
        if n_max(get_pts(mid)) >= 2: lo = mid
        else: hi = mid
    pm = [p for p, k in get_pts(hi) if k == 'max']; prev = pm[0] if pm else 0.0; pl = get_pts(lo)
    mx = sorted(p for p, k in pl if k == 'max'); mn = [p for p, k in pl if k == 'min']
    return float(lo), float(abs((mn[0] if mn else 0.5*(mx[0]+mx[-1])) - prev))
def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(784), delta); u /= np.linalg.norm(u)
    return u.astype(np.float32), (0.5*(mu_a+mu_b)).astype(np.float32), 0.5*float(delta @ u)
def balanced(X, Y, pair):
    Xa = X[Y == pair[0]]; Xb = X[Y == pair[1]]; n = min(len(Xa), len(Xb)); return Xa[:n], Xb[:n]
def cusp_fit(m, f, w, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(m) < window) & (w > 0); c3, c2, c1, c0 = np.polyfit(m[ok], f[ok], 3, w=np.sqrt(w[ok]))
    b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return dict(h=c0 + c1*s + c2*s**2 + c3*s**3, a=c1 + 3*c3*s**2 + 2*c2*s, b=b)
def analyse_pair(Xa, Xb):
    u, mid, half = lda_direction(Xa, Xb); m0 = np.concatenate([(Xa-mid) @ u, (Xb-mid) @ u]) / half
    bj = birth_bisect(lambda t: stationary_points(smooth_hist(m0, t))); return dict(tau=bj[0], jump=bj[1], m0=m0, u=u, mid=mid, half=half)

class ResBlock(nn.Module):
    def __init__(self, c, emb):
        super().__init__(); self.n1 = nn.GroupNorm(8, c); self.c1 = nn.Conv2d(c, c, 3, padding=1); self.n2 = nn.GroupNorm(8, c); self.c2 = nn.Conv2d(c, c, 3, padding=1); self.e = nn.Linear(emb, c)
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.e(e)[:, :, None, None]; return x + self.c2(F.silu(self.n2(h)))
class UNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__(); self.emb = nn.Sequential(nn.Linear(64, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1 = ResBlock(ch, emb); self.down = nn.Conv2d(ch, 2*ch, 3, stride=2, padding=1); self.d2 = ResBlock(2*ch, emb); self.mid = ResBlock(2*ch, emb)
        self.up = nn.ConvTranspose2d(2*ch, ch, 4, stride=2, padding=1); self.u1 = ResBlock(ch, emb); self.out = nn.Conv2d(ch, 1, 3, padding=1)
    def forward(self, x, c_noise):
        f = torch.exp(torch.arange(32, device=x.device) * -(math.log(1000)/31)); e = self.emb(torch.cat([torch.sin(c_noise[:, None]*f), torch.cos(c_noise[:, None]*f)], 1))
        h1 = self.d1(self.inp(x), e); h2 = self.mid(self.d2(self.down(h1), e), e); return self.out(self.u1(self.up(h2) + h1, e))
def denoise(net, x, sigma):
    s = sigma.view(-1, 1, 1, 1); c_skip = SIGMA_DATA**2/(s**2 + SIGMA_DATA**2); c_out = s*SIGMA_DATA/torch.sqrt(s**2 + SIGMA_DATA**2)
    return c_skip*x + c_out*net(x/torch.sqrt(s**2 + SIGMA_DATA**2), 0.25*torch.log(sigma))
def train(pair, seed, epochs=EPOCHS, bs=128, lr=2e-4, P_mean=-0.6, P_std=1.4, ema_decay=0.999):
    torch.manual_seed(seed); np.random.seed(seed)
    X = torch.from_numpy(X_MN[np.isin(Y_MN, pair)]).view(-1, 1, 28, 28).to(dev)
    net = UNet().to(dev); ema = copy.deepcopy(net); opt = torch.optim.Adam(net.parameters(), lr=lr); n = len(X); t0 = time.time(); out = {}
    for ep in range(1, epochs+1):
        perm = torch.randperm(n, device=dev)
        for i in range(0, n-bs+1, bs):
            x0 = X[perm[i:i+bs]]; sigma = torch.exp(P_mean + P_std*torch.randn(bs, device=dev))
            xt = x0 + sigma.view(-1,1,1,1)*torch.randn_like(x0); w = (sigma**2 + SIGMA_DATA**2)/(sigma*SIGMA_DATA)**2
            loss = (w.view(-1,1,1,1)*(denoise(net, xt, sigma) - x0)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                for pe, pn in zip(ema.parameters(), net.parameters()): pe.lerp_(pn, 1-ema_decay)
        if ep in (20, epochs):
            out[ep] = copy.deepcopy(ema).eval(); torch.save(out[ep].state_dict(), f'/kaggle/working/unet_{pair[0]}{pair[1]}_s{seed}_ep{ep}.pt')
    print(f"trained {pair} seed {seed} in {time.time()-t0:.0f}s"); return out

NETS = {}
for pair in PAIRS:
    for seed in (SEEDS if pair != (3, 8) else [0]):
        NETS[(pair, seed)] = train(pair, seed)

## 11A. Bisected network transitions (corrected convention)

In [ ]:
def zero_crossings(cen, mean, cnt, min_count=150):
    ok = cnt >= min_count; c, f = cen[ok], mean[ok]; out = []
    for i in range(len(c)-1):
        if np.sign(f[i]) != np.sign(f[i+1]) and f[i] != 0: out.append((float(c[i]-f[i]*(c[i+1]-c[i])/(f[i+1]-f[i])), 'max' if f[i] > 0 else 'min'))
    return _merge(out, merge=0.05)
@torch.no_grad()
def net_binned(net, Xflat, u, mid, half, tau, n=30000, nbins=70):
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); sigma = half*np.sqrt(tau)
    x0 = torch.from_numpy(Xflat[rng.choice(len(Xflat), n)]).to(dev); xt = x0 + sigma*torch.randn_like(x0)
    D = torch.cat([denoise(net, xb.view(-1,1,28,28), torch.full((len(xb),), sigma, device=dev)).view(len(xb), -1) for xb in xt.split(2000)])
    m = ((xt-midt) @ ut / half).cpu().numpy(); sm = (half*((D-xt) @ ut)/sigma**2).cpu().numpy()
    lo, hi = np.quantile(m, [0.005, 0.995]); edges = np.linspace(lo, hi, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
    cnt = np.bincount(b, minlength=nbins).astype(float); s = np.bincount(b, weights=sm, minlength=nbins); k = np.exp(-0.5*(np.arange(-5, 6)/1.5)**2)
    return 0.5*(edges[:-1]+edges[1:]), np.convolve(s, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9), cnt
def net_points(*a, **k): return zero_crossings(*net_binned(*a, **k))

R = {pair: analyse_pair(*balanced(X_MN, Y_MN, pair)) for pair in PAIRS}
print(f"{'pair':>6} {'seed':>4} {'ep':>4} {'exact tau_bif':>14} {'net tau_bif':>12} {'net jump':>9} {'exact jump':>11}")
NETBIF = {}
for (pair, seed), ck in NETS.items():
    r = R[pair]; Xflat = np.concatenate(balanced(X_MN, Y_MN, pair))
    for ep, net in ck.items():
        bj = birth_bisect(lambda t: net_points(net, Xflat, r['u'], r['mid'], r['half'], t), taus=np.geomspace(0.1, 3.0, 36)); NETBIF[(pair, seed, ep)] = bj
        print(f"{pair!s:>6} {seed:4d} {ep:4d} {r['tau']:14.3f} {(bj[0] if bj else float('nan')):12.3f} {(bj[1] if bj else float('nan')):9.2f} {r['jump']:11.2f}")
if len(SEEDS) > 1:
    for pair in [(0, 1), (0, 8)]:
        for ep in (20, EPOCHS):
            v = [NETBIF[(pair, s, ep)][0] for s in SEEDS if NETBIF.get((pair, s, ep))]
            print(f"{pair} ep{ep}: net tau_bif = {np.mean(v):.3f} +/- {np.std(v):.3f} (n={len(v)})  exact {R[pair]['tau']:.3f}")

## 12F. Causal interventions on the network's excess asymmetry

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__(); self.c1 = nn.Conv2d(1, 32, 3, padding=1); self.c2 = nn.Conv2d(32, 64, 3, padding=1); self.fc = nn.Linear(64*7*7, 2)
    def forward(self, x): x = F.max_pool2d(F.relu(self.c1(x)), 2); x = F.max_pool2d(F.relu(self.c2(x)), 2); return self.fc(x.flatten(1))
def train_cnn(Xa, Xb, epochs=3):
    X = torch.from_numpy(np.concatenate([Xa, Xb])).view(-1, 1, 28, 28).to(dev); Y = torch.cat([torch.zeros(len(Xa)), torch.ones(len(Xb))]).long().to(dev)
    net = CNN().to(dev); opt = torch.optim.Adam(net.parameters(), 1e-3)
    for _ in range(epochs):
        perm = torch.randperm(len(X), device=dev)
        for i in range(0, len(X), 256): idx = perm[i:i+256]; loss = F.cross_entropy(net(X[idx]), Y[idx]); opt.zero_grad(); loss.backward(); opt.step()
    net.eval(); return net
@torch.no_grad()
def sample(net, u, mid, half, hcorr, steps=40, churn=0.0, n=2000, s_max=20.0, s_min=0.002, rho=7.0):
    ut = torch.from_numpy(u).to(dev).view(1, 1, 28, 28)
    def Dc(x, s):
        D = denoise(net, x, s.expand(n)); tau = float(s)**2/half**2; return D - (float(s)**2*hcorr(tau)/half)*ut
    i = torch.arange(steps, device=dev); sig = torch.cat([(s_max**(1/rho) + i/(steps-1)*(s_min**(1/rho)-s_max**(1/rho)))**rho, torch.zeros(1, device=dev)])
    x = torch.randn(n, 1, 28, 28, device=dev)*sig[0]
    for k in range(steps):
        s0, s1 = sig[k], sig[k+1]
        if churn > 0 and 0.01 <= s0 <= s_max:
            gam = min(churn/steps, 2**0.5-1); sh = s0*(1+gam); x = x + torch.sqrt(sh**2 - s0**2)*torch.randn_like(x); s0 = sh
        d0 = (x - Dc(x, s0))/s0; x1 = x + (s1-s0)*d0
        if s1 > 0: d1 = (x1 - Dc(x1, s1))/s1; x = x + (s1-s0)*0.5*(d0+d1)
        else: x = x1
    return x

print(f"{'pair':>6} {'ep':>4} {'intervention':>16} {'sampler':>8} {'frac+ (LDA)':>12} {'frac+ (CNN)':>12}")
ROWS = []
for pair in [(0, 1), (0, 8)]:
    Xa, Xb = balanced(X_MN, Y_MN, pair); r = R[pair]; Xflat = np.concatenate([Xa, Xb]); u, mid, half = r['u'], r['mid'], r['half']
    cnn = train_cnn(Xa, Xb); ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev)
    real = torch.from_numpy(Xflat).view(-1, 1, 28, 28).to(dev); print(f"   CNN agreement with true labels on real data: {((cnn(real[:4000]).argmax(1) == 0).float().mean()*2):.3f} (should be ~1.0 for class a half)")
    for ep in (20, EPOCHS):
        net = NETS[(pair, 0)][ep]
        taus = np.geomspace(0.2, 6.0, 20); hn = np.array([cusp_fit(*net_binned(net, Xflat, u, mid, half, t, n=20000))['h'] for t in taus]); hx = np.array([cusp_fit(*exact_drift(r['m0'], t))['h'] for t in taus])
        dh = hn - hx
        def mk(lo, hi, dh=dh): return lambda tau: float(np.interp(np.log(tau), np.log(taus), dh)) if lo <= tau <= hi else 0.0
        INT = {'none': lambda tau: 0.0, 'window[0.3,1.5]': mk(0.3, 1.5), 'high-noise>1.5': mk(1.5, 6.0), 'both': mk(0.3, 6.0)}
        print(f"   {pair} ep{ep}: excess h_net-h_exact, window mean {np.mean(dh[(taus>=0.3)&(taus<=1.5)]):+.3f}, high-noise mean {np.mean(dh[taus>1.5]):+.3f}")
        for name, hc in INT.items():
            for sname, kw in [('heun40', dict(steps=40)), ('sde10', dict(steps=10, churn=40)), ('sde20', dict(steps=20, churn=40)), ('sde40', dict(steps=40, churn=40)), ('sde80', dict(steps=80, churn=40))]:
                x = sample(net, u, mid, half, hc, **kw); fl = float((((x.view(-1, 784)-midt) @ ut / half) > 0).float().mean()); fc = float((cnn(x).argmax(1) == 0).float().mean())
                ROWS.append((pair, ep, name, sname, fl, fc)); print(f"{pair!s:>6} {ep:4d} {name:>16} {sname:>8} {fl:12.3f} {fc:12.3f}")

## What to report back
1. The 11A table (and seed means if `SEEDS=[0,1,2]`).
2. The 12F table, including the CNN-agreement line and the "excess h" lines. Predictions: window correction removes the ep20 bias; high-noise correction removes the ep80 residual; both give ≈0.50 under both label sets and all samplers.